# Examen 2
> Moctezuma Ramirez Diego Rafael

## Carga de modulos

In [1]:
import warnings
from sklearn.exceptions import ConvergenceWarning

# Ignorar los mensajes de ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
import pandas as pd
import numpy as np

import plotly.express as px

# Data preprocessing
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import  StandardScaler, OneHotEncoder, MinMaxScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Modeling
from sklearn.model_selection import cross_val_score, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet, Lars, Lasso, BayesianRidge, SGDRegressor

#pd.set_option('display.float_format', lambda x: '%.6f' % x)

## Funciones auxiliares

In [3]:
def frecuencias(df, caracteristicas):
    for caracteristica in caracteristicas:
        print(f"Caracteristica: {caracteristica}")
        abs_ = df[caracteristica].value_counts(dropna=False).to_frame().rename(columns={"count": "Frecuencia absoluta"})
        rel_ = df[caracteristica].value_counts(dropna=False, normalize= True).to_frame().rename(columns={"proportion": "Frecuencia relativa"})
        freq = abs_.join(rel_)
        freq["Frecuencia Acumulada"] = freq["Frecuencia absoluta"].cumsum()
        freq["Acumulada %"] = freq["Frecuencia relativa"].cumsum()
        freq["Frecuencia absoluta"] = freq["Frecuencia absoluta"].map(lambda x: f"{x:,.0f}")
        freq["Frecuencia relativa"] = freq["Frecuencia relativa"].map(lambda x: f"{x:,.2%}")
        freq["Frecuencia Acumulada"] = freq["Frecuencia Acumulada"].map(lambda x: f"{x:,.0f}")
        freq["Acumulada %"] = freq["Acumulada %"].map(lambda x: f"{x:,.2%}")
        display(freq)

## Carga de datos con pandas

In [4]:
df = pd.read_csv("DataRegresion/train_PAY_AMT1.csv",sep='|')

In [5]:
df.sample(5)

,CUSTOMER_ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
3693,26983,420000.0,2,2,2,27,-2,-2,-2,-2,...,15000.0,15000.0,15000.0,15000.0,15000.0,15000.0,15000.0,15000.0,15000.0,14996.0
927,24657,180000.0,1,1,2,28,0,0,0,0,...,30820.0,23852.0,17902.0,7784.0,1560.0,1735.0,1385.0,30.0,23.0,5411.0
3069,6521,200000.0,2,3,1,29,2,2,2,0,...,132120.0,128669.0,131499.0,143020.0,6000.0,4600.0,0.0,4800.0,13700.0,500.0
754,22038,100000.0,2,1,1,42,2,2,2,2,...,46112.0,48925.0,49794.0,50823.0,2000.0,0.0,3900.0,1800.0,2000.0,2000.0
4901,8532,50000.0,2,2,2,44,0,0,0,-1,...,47608.0,3990.0,390.0,0.0,2000.0,1000.0,3600.0,390.0,0.0,4870.0


In [6]:
df.dtypes

CUSTOMER_ID      int64
LIMIT_BAL      float64
SEX              int64
EDUCATION        int64
MARRIAGE         int64
AGE              int64
PAY_2            int64
PAY_3            int64
PAY_4            int64
PAY_5            int64
PAY_6            int64
BILL_AMT1      float64
BILL_AMT2      float64
BILL_AMT3      float64
BILL_AMT4      float64
BILL_AMT5      float64
BILL_AMT6      float64
PAY_AMT1       float64
PAY_AMT2       float64
PAY_AMT3       float64
PAY_AMT4       float64
PAY_AMT5       float64
PAY_AMT6       float64
dtype: object

In [7]:
df.isna().sum()

CUSTOMER_ID    0
LIMIT_BAL      0
SEX            0
EDUCATION      0
MARRIAGE       0
AGE            0
PAY_2          0
PAY_3          0
PAY_4          0
PAY_5          0
PAY_6          0
BILL_AMT1      0
BILL_AMT2      0
BILL_AMT3      0
BILL_AMT4      0
BILL_AMT5      0
BILL_AMT6      0
PAY_AMT1       0
PAY_AMT2       0
PAY_AMT3       0
PAY_AMT4       0
PAY_AMT5       0
PAY_AMT6       0
dtype: int64

## EDA

### Filtrado de variables

In [8]:
ls_discretas = ["SEX","EDUCATION","MARRIAGE","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]
ls_continuas = ["BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6","PAY_AMT2","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6","LIMIT_BAL","AGE"]
ls_drop = ["CUSTOMER_ID"]

target = "PAY_AMT1"

In [9]:
df_customer_ids = df["CUSTOMER_ID"]
df.drop(columns=ls_drop, inplace=True)

### EDA Discreto

In [10]:
frecuencias(df, ls_discretas)

Caracteristica: SEX


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
SEX,,,,
2,"3,409",60.60%,"3,409",60.60%
1,"2,216",39.40%,"5,625",100.00%


Caracteristica: EDUCATION


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
EDUCATION,,,,
2,"2,578",45.83%,"2,578",45.83%
1,"1,994",35.45%,"4,572",81.28%
3,979,17.40%,"5,551",98.68%
5,42,0.75%,"5,593",99.43%
4,20,0.36%,"5,613",99.79%
6,10,0.18%,"5,623",99.96%
0,2,0.04%,"5,625",100.00%


Caracteristica: MARRIAGE


...

,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
PAY_4,,,,
0,"3,087",54.88%,"3,087",54.88%
-1,"1,101",19.57%,"4,188",74.45%
-2,774,13.76%,"4,962",88.21%
2,606,10.77%,"5,568",98.99%
3,25,0.44%,"5,593",99.43%
7,13,0.23%,"5,606",99.66%
5,9,0.16%,"5,615",99.82%
4,8,0.14%,"5,623",99.96%
1,1,0.02%,"5,624",99.98%


Caracteristica: PAY_5


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
PAY_5,,,,
0,"3,216",57.17%,"3,216",57.17%
-1,"1,075",19.11%,"4,291",76.28%
-2,788,14.01%,"5,079",90.29%
2,493,8.76%,"5,572",99.06%
3,24,0.43%,"5,596",99.48%
7,13,0.23%,"5,609",99.72%
4,12,0.21%,"5,621",99.93%
5,4,0.07%,"5,625",100.00%


Caracteristica: PAY_6


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
PAY_6,,,,
0,"3,091",54.95%,"3,091",54.95%
-1,"1,120",19.91%,"4,211",74.86%
-2,860,15.29%,"5,071",90.15%
2,501,8.91%,"5,572",99.06%
3,29,0.52%,"5,601",99.57%
7,13,0.23%,"5,614",99.80%
4,7,0.12%,"5,621",99.93%
6,3,0.05%,"5,624",99.98%
5,1,0.02%,"5,625",100.00%


### EDA Continuas

In [11]:
for caracteristica in ls_continuas:
    fig = px.histogram(df, x=caracteristica, nbins=50, title=f"Histograma de {caracteristica}",template = "plotly_dark")
    fig.show()

### Feature Engineering

In [12]:
df['TOTAL_DEBT'] = df[['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']].sum(axis=1)
df['USAGE_RATIO'] = df['TOTAL_DEBT'] / df['LIMIT_BAL']
df['TOTAL_PAYMENTS'] = df[['PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']].sum(axis=1)
df['PAY_AMT_MEAN'] = df[['PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']].mean(axis=1)
df['PAY_STA_MEAN'] = df[['PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']].mean(axis=1).round(2)

#### Normalización

> EDUCATION

In [13]:
redundantes_e = [0,5,6]
df['EDUCATION'] = df['EDUCATION'].replace(redundantes_e, 4)

In [14]:
df["EDUCATION"].value_counts(True)

EDUCATION
2    0.458311
1    0.354489
3    0.174044
4    0.013156
Name: proportion, dtype: float64

> MARRIAGE

In [15]:
redundantes_m = [0]
df['MARRIAGE'] = df['MARRIAGE'].replace(redundantes_m, 3)

In [16]:
df["MARRIAGE"].value_counts(True)

MARRIAGE
2    0.529422
1    0.459556
3    0.011022
Name: proportion, dtype: float64

In [17]:
df[ls_discretas].sample(5)

,SEX,EDUCATION,MARRIAGE,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6
2468,1,2,1,-1,-2,-2,-2,-2
1615,1,2,2,-1,0,0,-1,-1
5423,2,2,2,-1,-1,-1,0,0
3435,2,3,2,0,0,0,0,0
4154,2,2,1,2,0,0,0,0


### Data Cleaning

In [18]:
shape_out = df.shape

In [19]:
df[ls_continuas].describe(percentiles=[0.01, 0.05, 0.95, 0.99])

,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,LIMIT_BAL,AGE
count,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000,5.625000e+03,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000
mean,51767.287644,49310.678933,46993.737778,43279.012622,40108.183644,38976.025600,6.060180e+03,5479.124800,4868.507022,5040.530133,5463.827378,166940.444444,35.614578
std,73436.287748,70475.318499,67915.899619,62893.586019,58990.542351,58881.448383,2.257444e+04,18028.872953,15830.136064,15435.918866,18808.832708,129162.999951,9.380835
min,-10682.000000,-69777.000000,-157264.000000,-170000.000000,-61372.000000,-339603.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,10000.000000,21.000000
1%,-73.560000,-199.040000,-200.000000,-212.000000,-215.800000,-296.400000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,10000.000000,22.000000
5%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,20000.000000,23.000000
50%,22369.000000,21222.000000,20237.000000,19170.000000,18121.000000,17352.000000,2.006000e+03,1800.000000,1500.000000,1600.000000,1500.000000,140000.000000,34.000000
95%,204435.200000,196052.000000,187822.200000,171860.200000,164673.800000,162118.000000,1.825380e+04,18569.600000,15896.600000,18178.200000,18127.600000,430000.000000,53.000000
99%,332624.520000,321595.920000,306782.640000,285670.560000,271680.560000,267570.080000,7.880608e+04,78325.440000,65880.080000,67350.760000,84657.520000,500000.000000,61.000000
max,588000.000000,597793.000000,855086.000000,569034.000000,551702.000000,699944.000000,1.024516e+06,508229.000000,497000.000000,426529.000000,528666.000000,750000.000000,75.000000


In [20]:
dic_outliers = {}
for caracteristica in ls_continuas:
    aux = df[caracteristica].quantile(0.99) + df[caracteristica].std() * 4
    max_val = df[caracteristica].max()
    if max_val > aux:
        dic_outliers[caracteristica] = df[caracteristica].quantile(0.99)

In [21]:
dic_outliers

{'BILL_AMT3': np.float64(306782.640000001),
 'BILL_AMT4': np.float64(285670.56000000006),
 'BILL_AMT5': np.float64(271680.56),
 'BILL_AMT6': np.float64(267570.08),
 'PAY_AMT2': np.float64(78806.08000000018),
 'PAY_AMT3': np.float64(78325.44000000005),
 'PAY_AMT4': np.float64(65880.08000000013),
 'PAY_AMT5': np.float64(67350.7600000002),
 'PAY_AMT6': np.float64(84657.52000000031)}

In [22]:
for caracteristica, perc in dic_outliers.items():
    df = df[(df[caracteristica] <= perc) | (df[caracteristica].isna())]

In [23]:
df.shape[0] / shape_out[0] 

0.9448888888888889

#### Variables altamente vacías

In [24]:
df.isna().mean().sort_values()

LIMIT_BAL         0.0
TOTAL_PAYMENTS    0.0
USAGE_RATIO       0.0
TOTAL_DEBT        0.0
PAY_AMT6          0.0
PAY_AMT5          0.0
PAY_AMT4          0.0
PAY_AMT3          0.0
PAY_AMT2          0.0
PAY_AMT1          0.0
BILL_AMT6         0.0
BILL_AMT5         0.0
PAY_AMT_MEAN      0.0
BILL_AMT4         0.0
BILL_AMT2         0.0
BILL_AMT1         0.0
PAY_6             0.0
PAY_5             0.0
PAY_4             0.0
PAY_3             0.0
PAY_2             0.0
AGE               0.0
MARRIAGE          0.0
EDUCATION         0.0
SEX               0.0
BILL_AMT3         0.0
PAY_STA_MEAN      0.0
dtype: float64

#### Variables unarias

In [25]:
df.nunique()

LIMIT_BAL           61
SEX                  2
EDUCATION            4
MARRIAGE             3
AGE                 53
PAY_2               10
PAY_3               10
PAY_4               10
PAY_5                8
PAY_6                9
BILL_AMT1         4595
BILL_AMT2         4531
BILL_AMT3         4449
BILL_AMT4         4389
BILL_AMT5         4303
BILL_AMT6         4248
PAY_AMT1          2229
PAY_AMT2          2174
PAY_AMT3          2111
PAY_AMT4          1950
PAY_AMT5          1961
PAY_AMT6          1898
TOTAL_DEBT        5066
USAGE_RATIO       5126
TOTAL_PAYMENTS    3964
PAY_AMT_MEAN      3964
PAY_STA_MEAN        35
dtype: int64

In [26]:
df.std()

LIMIT_BAL         122296.847303
SEX                    0.487929
EDUCATION              0.746826
MARRIAGE               0.519387
AGE                    9.426170
PAY_2                  1.196499
PAY_3                  1.193324
PAY_4                  1.179498
PAY_5                  1.127642
PAY_6                  1.149974
BILL_AMT1          61321.353212
BILL_AMT2          58596.432781
BILL_AMT3          53939.670535
BILL_AMT4          50211.749545
BILL_AMT5          47564.604574
BILL_AMT6          46530.583860
PAY_AMT1           12220.015460
PAY_AMT2            6960.712272
PAY_AMT3            7153.455138
PAY_AMT4            6438.254720
PAY_AMT5            6568.461070
PAY_AMT6            7424.028232
TOTAL_DEBT        304841.053324
USAGE_RATIO            2.105744
TOTAL_PAYMENTS     22082.388200
PAY_AMT_MEAN        4416.477640
PAY_STA_MEAN           1.014983
dtype: float64

### Selección mejores variables

In [27]:
kb = SelectKBest(k="all", score_func=f_regression)

In [28]:
columnas = df.columns.to_list()
columnas.remove(target)
X = df[columnas]
y = df[target]

In [29]:
kb.fit(X, y)

,score_func,<function f_r...t 0x12fac81f0>
,k,'all'


In [30]:
df_scores = pd.DataFrame(data=zip(X.columns, kb.scores_), columns=["feature", "score"]).set_index("feature").sort_values(by="score", ascending=False)

In [31]:
df_scores.head(20)

,score
feature,
PAY_AMT_MEAN,614.157717
TOTAL_PAYMENTS,614.157717
BILL_AMT2,469.513086
PAY_AMT2,378.350906
BILL_AMT3,348.536930
BILL_AMT4,315.193048
TOTAL_DEBT,302.967931
PAY_AMT3,289.239425
BILL_AMT5,278.453552


In [32]:
fig = px.bar(df_scores.head(50), template = "plotly_dark")
fig.show()

## Modelado

### Entrenamiento del modelo

#### Standard Escaler

In [33]:
ss = StandardScaler()
#ss = RobustScaler()
#ss = MinMaxScaler()

#### Regresión Lineal

In [34]:
lg = LinearRegression()

#### LARS

In [35]:
lrs = Lars()

#### LASSO

In [36]:
lasso = Lasso()

#### Ridge

In [37]:
ridge = Ridge()

#### Elastic Net

In [38]:
elan = ElasticNet()

#### Red Bayesiana

In [39]:
redbay = BayesianRidge()

#### Gradiente Estocástico Descendiente

In [40]:
ged = SGDRegressor()

### Comparación de modelos

#### Usando selectKBest

In [41]:
modelos = {
    "Regresion Lineal": lg,
    "Lars": lrs,
    "Lasso": lasso,
    "Ridge": ridge,
    "Elastic Net": elan,
    "Red Bayesiana": redbay,
    "Gradiente Estocástico": ged
}

In [42]:
df_kbest_model = pd.DataFrame(columns=["Regresion Lineal", "Lars", "Lasso", "Ridge", "Elastic Net", "Red Bayesiana", "Gradiente Estocástico"])
X_train, X_test, y_train, y_test = train_test_split(X, y)

y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

In [43]:
transformer = ColumnTransformer([
    ("scaler", ss, ls_continuas),
    ("dummies", OneHotEncoder(handle_unknown="ignore"), ls_discretas)
])

In [44]:
for k in range(5, 61, 5):
    resultados_k = {}   

    for nombre, modelo in modelos.items():
        pipe = Pipeline([
            ("prep", transformer),
            ("kbest", SelectKBest(f_regression, k=k)),
            ("modelo", modelo)
        ])
        pipe.fit(X_train, y_train)
        score = pipe.score(X_test, y_test)
        resultados_k[nombre] = score

    df_kbest_model.loc[k] = resultados_k

In [45]:
df_kbest_model

,Regresion Lineal,Lars,Lasso,Ridge,Elastic Net,Red Bayesiana,Gradiente Estocástico
5,0.115494,1.154944e-01,0.115628,0.115458,0.089383,0.115256,0.111837
10,0.137171,1.368039e-01,0.137433,0.137422,0.130927,0.139626,0.143328
15,0.246286,2.462861e-01,0.246529,0.246613,0.145108,0.246975,0.247594
20,0.249032,2.490319e-01,0.249337,0.249394,0.148599,0.250017,0.248509
25,0.251675,2.516748e-01,0.251860,0.252008,0.148473,0.252774,0.246054
30,0.252094,2.520938e-01,0.252379,0.252468,0.148777,0.253518,0.239323
35,0.252423,2.582326e-01,0.252780,0.252821,0.148922,0.253894,0.245982
40,0.253036,2.585500e-01,0.253329,0.253423,0.148790,0.254563,0.246568
45,0.252041,2.570682e-01,0.252417,0.252350,0.148363,0.253484,0.250792
50,0.252042,2.431548e-01,0.252745,0.252503,0.148639,0.253563,0.250944


In [46]:
max_lr = [df_kbest_model["Regresion Lineal"].idxmax(),df_kbest_model["Regresion Lineal"].max()]
max_lars = [df_kbest_model["Lars"].idxmax(),df_kbest_model["Lars"].max()]
max_lasso = [df_kbest_model["Lasso"].idxmax(),df_kbest_model["Lasso"].max()]
max_ridge = [df_kbest_model["Ridge"].idxmax(),df_kbest_model["Ridge"].max()]
max_elnet = [df_kbest_model["Elastic Net"].idxmax(),df_kbest_model["Elastic Net"].max()]
max_redbay = [df_kbest_model["Red Bayesiana"].idxmax(),df_kbest_model["Red Bayesiana"].max()]
max_ged = [df_kbest_model["Gradiente Estocástico"].idxmax(),df_kbest_model["Gradiente Estocástico"].max()]

In [47]:
df_modelos = pd.DataFrame(columns=["modelo","mejor_k","mejor_score"])

df_modelos.loc[0] = ["Linear Regression", max_lr[0], max_lr[1]]
df_modelos.loc[1] = ["LARS", max_lars[0], max_lars[1]]
df_modelos.loc[2] = ["Lasso", max_lasso[0], max_lasso[1]]
df_modelos.loc[3] = ["Ridge", max_ridge[0], max_ridge[1]]
df_modelos.loc[4] = ["Elastic Net", max_elnet[0], max_elnet[1]]
df_modelos.loc[5] = ["Red Bayesiana", max_redbay[0], max_redbay[1]]
df_modelos.loc[6] = ["Gradiente Estocástico", max_ged[0], max_ged[1]]

In [48]:
df_modelos.sort_values(by="mejor_score", ascending=False)

,modelo,mejor_k,mejor_score
1,LARS,40,0.258550
5,Red Bayesiana,40,0.254563
3,Ridge,40,0.253423
2,Lasso,40,0.253329
0,Linear Regression,40,0.253036
6,Gradiente Estocástico,50,0.250944
4,Elastic Net,35,0.148922


##### Cross validation 

In [49]:
df_crossv = pd.DataFrame(columns = ["modelo","f1","f2","f3","f4","f5","mean","std"])

In [50]:
dic_cross = {
    "Regresion lineal": [lg, max_lr[0]],
    "LARS": [lrs, max_lars[0]],
    "Lasso": [lasso, max_lasso[0]],
    "Ridge": [ridge, max_ridge[0]],
    "Elastic Net": [elan, max_elnet[0]],
    "Red Bayesiana": [redbay, max_redbay[0]],
    "Gradiente Estocástico": [ged, max_ged[0]]
}

In [51]:
for nombre_modelo, info in dic_cross.items():
    pipe = Pipeline([
        ("prep", transformer),
        ("kbest", SelectKBest(score_func=f_regression, k=info[1])),
        ("modelo", info[0])
    ])

    cross = cross_val_score(estimator=pipe,X=X,y=y.values.ravel(),cv=5)

    df_crossv.loc[len(df_crossv)] = [nombre_modelo,cross[0],cross[1],cross[2],cross[3],cross[4],cross.mean(),cross.std()    ]

In [52]:
df_crossv

,modelo,f1,f2,f3,f4,f5,mean,std
0,Regresion lineal,0.422708,0.326184,0.267040,0.342234,0.294603,0.330554,0.052867
1,LARS,0.424345,0.327596,0.263795,0.341165,0.294208,0.330222,0.054213
2,Lasso,0.424020,0.327212,0.267186,0.341456,0.294836,0.330942,0.053210
3,Ridge,0.423908,0.328121,0.267305,0.342071,0.294754,0.331232,0.053166
4,Elastic Net,0.206204,0.206383,0.123773,0.194335,0.143075,0.174754,0.034570
5,Red Bayesiana,0.422433,0.331378,0.268125,0.342947,0.294982,0.331973,0.052438
6,Gradiente Estocástico,0.412654,0.312013,0.271297,0.342655,0.299510,0.327626,0.048297


##### Comparación de cross validations

In [53]:
df_crossv[["modelo","mean"]].sort_values(by="mean", ascending=False)

,modelo,mean
5,Red Bayesiana,0.331973
3,Ridge,0.331232
2,Lasso,0.330942
0,Regresion lineal,0.330554
1,LARS,0.330222
6,Gradiente Estocástico,0.327626
4,Elastic Net,0.174754


## Hiperparametrización

In [54]:
param = {
    "Regresion lineal": {
        "modelo__fit_intercept": [True, False],
        "modelo__positive": [False, True]
    },

    "LARS": {
        "modelo__fit_intercept": [True, False],
        "modelo__n_nonzero_coefs": [5, 10, 20, 50, 100, np.inf]
    },

    "Lasso": {
        "modelo__alpha": [0.001, 0.01, 0.1, 1.0, 10.0],
        "modelo__max_iter": [1000, 5000, 10000]
    },

    "Ridge": {
        "modelo__alpha": [0.001, 0.01, 0.1, 1.0, 10.0],
        "modelo__solver": ["auto", "svd", "cholesky", "lsqr"]
    },

    "Elastic Net": {
        "modelo__alpha": [0.001, 0.01, 0.1, 1.0, 10.0],
        "modelo__l1_ratio": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        "modelo__max_iter": [1000, 5000, 10000]
    },

    "Red Bayesiana": {
        "modelo__alpha_1": [1e-6, 1e-4, 1e-2],
        "modelo__alpha_2": [1e-6, 1e-4, 1e-2],
        "modelo__lambda_1": [1e-6, 1e-4, 1e-2],
        "modelo__lambda_2": [1e-6, 1e-4, 1e-2]
    },

    "Gradiente Estocástico": {
        "modelo__loss": ["squared_error", "huber"],
        "modelo__penalty": ["l2", "l1", "elasticnet"],
        "modelo__alpha": [0.0001, 0.001, 0.01],
        "modelo__learning_rate": ["constant", "optimal", "invscaling", "adaptive"],
        "modelo__eta0": [0.001, 0.01, 0.1],
        "modelo__max_iter": [1000, 5000, 10000]
    }
}

In [55]:
df_hyper = pd.DataFrame(columns=["Modelo", "Mejor score", "Std", "Parametros buenos"])

for modelo, info in dic_cross.items():
    pipe = Pipeline([
        ("prep", transformer),
        ("kbest", SelectKBest(f_regression, k=info[1])),
        ("modelo", info[0])
    ])

    grid = param[modelo]
    search = RandomizedSearchCV(estimator=pipe, param_distributions=grid, scoring="r2", n_iter=20, cv=5, n_jobs=-1, random_state=777)

    search.fit(X_train, y_train)
    df_hyper.loc[len(df_hyper)] = [modelo,search.best_score_,search.cv_results_['std_test_score'][search.best_index_],search.best_params_]

/opt/anaconda3/envs/CD/lib/python3.10/site-packages/sklearn/model_selection/_search.py:317: UserWarning:

The total space of parameters 4 is smaller than n_iter=20. Running 4 iterations. For exhaustive searches, use GridSearchCV.

/opt/anaconda3/envs/CD/lib/python3.10/site-packages/sklearn/model_selection/_search.py:317: UserWarning:

The total space of parameters 12 is smaller than n_iter=20. Running 12 iterations. For exhaustive searches, use GridSearchCV.

/opt/anaconda3/envs/CD/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning:


10 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File

In [56]:
df_hyper.sort_values(by="Mejor score", ascending=False)

,Modelo,Mejor score,Std,Parametros buenos
6,Gradiente Estocástico,0.326284,0.059736,"{'modelo__penalty': 'l1', 'modelo__max_iter': ..."
3,Ridge,0.326066,0.063910,"{'modelo__solver': 'svd', 'modelo__alpha': 10.0}"
4,Elastic Net,0.326048,0.060430,"{'modelo__max_iter': 1000, 'modelo__l1_ratio':..."
5,Red Bayesiana,0.323909,0.066926,"{'modelo__lambda_2': 0.0001, 'modelo__lambda_1..."
2,Lasso,0.322746,0.067772,"{'modelo__max_iter': 1000, 'modelo__alpha': 10.0}"
0,Regresion lineal,0.322338,0.070137,"{'modelo__positive': False, 'modelo__fit_inter..."
1,LARS,0.321491,0.060670,"{'modelo__n_nonzero_coefs': 20, 'modelo__fit_i..."


## Predicción

### Mejor modelo

#### Standard - Select K Best

In [57]:
nombre_modelo = df_crossv.loc[df_crossv["mean"].idxmax(), "modelo"]
info_modelo = dic_cross[nombre_modelo]
nombre_modelo

'Red Bayesiana'

In [58]:
modelo_ss = info_modelo[0]
mejor_k = info_modelo[1]

In [59]:
pipe_ss = Pipeline([
    ("prep", transformer),
    ("kbest", SelectKBest(f_regression, k=mejor_k)),
    ("modelo", modelo_ss)
])

In [60]:
pipe_ss.fit(X, y)

,steps,"[('prep', ...), ('kbest', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('dummies', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


#### Hiperparametrización

In [61]:
df_hyper["Parametros buenos"][df_hyper["Modelo"] == "Elastic Net"].values[0]

{'modelo__max_iter': 1000, 'modelo__l1_ratio': 0.4, 'modelo__alpha': 0.01}

In [62]:
modelo_h = ElasticNet(max_iter=1000,l1_ratio=0.0, alpha=0.01)
mejor_k = max_elnet[0]

In [63]:
pipe_h = Pipeline([
    ("prep", transformer),
    ("kbest", SelectKBest(f_regression, k=mejor_k)),
    ("modelo", modelo_h)
])

In [64]:
pipe_h.fit(X, y)

,steps,"[('prep', ...), ('kbest', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('dummies', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Carga de datos

In [65]:
df_predic = pd.read_csv("DataRegresion/val_PAY_AMT1.csv",sep='|')

In [66]:
df_predic

,CUSTOMER_ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
0,17391,210000.0,1,2,2,31,2,0,0,2,...,195530.0,176634.0,128137.0,130500.0,128838.0,6800.0,4900.0,4500.0,4700.0,4900.0
1,14756,30000.0,1,2,1,39,2,2,2,2,...,16652.0,16087.0,17317.0,16900.0,17465.0,0.0,1500.0,0.0,1000.0,3500.0
2,26188,410000.0,1,1,1,41,0,0,0,0,...,409780.0,383225.0,396283.0,274224.0,321787.0,20000.0,20000.0,50000.0,30000.0,43312.0
3,20929,140000.0,2,2,2,24,2,2,2,2,...,51916.0,52903.0,53652.0,52201.0,55497.0,2400.0,2200.0,0.0,4300.0,2200.0
4,6533,290000.0,2,2,1,49,0,0,0,0,...,36927.0,39769.0,35058.0,36965.0,38863.0,3500.0,2500.0,2500.0,2500.0,2000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1870,12040,30000.0,2,2,2,23,2,2,0,0,...,21050.0,20414.0,22008.0,23480.0,23777.0,0.0,1934.0,1824.0,830.0,0.0
1871,3318,300000.0,2,3,2,29,-2,-2,-2,-2,...,28092.0,3572.0,195.0,209.0,990.0,3589.0,195.0,210.0,994.0,0.0
1872,2124,330000.0,2,1,1,41,-1,-2,-2,-2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1873,13129,210000.0,2,2,1,33,-1,-1,-2,-2,...,748.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Normalización

> EDUCATION

In [67]:
redundantes_e = [0,5,6]
df_predic['EDUCATION'] = df_predic['EDUCATION'].replace(redundantes_e, 4)

> MARRIAGE

In [68]:
redundantes_m = [0]
df_predic['MARRIAGE'] = df_predic['MARRIAGE'].replace(redundantes_m, 3)

### Limpieza

In [69]:
shape_viejo = df_predic.shape

In [70]:
dic_outliers

{'BILL_AMT3': np.float64(306782.640000001),
 'BILL_AMT4': np.float64(285670.56000000006),
 'BILL_AMT5': np.float64(271680.56),
 'BILL_AMT6': np.float64(267570.08),
 'PAY_AMT2': np.float64(78806.08000000018),
 'PAY_AMT3': np.float64(78325.44000000005),
 'PAY_AMT4': np.float64(65880.08000000013),
 'PAY_AMT5': np.float64(67350.7600000002),
 'PAY_AMT6': np.float64(84657.52000000031)}

In [71]:
for caracteristica, perc in dic_outliers.items():
    df_predic = df_predic[(df_predic[caracteristica] <= perc) | (df_predic[caracteristica].isna())]

In [72]:
df_predic.shape[0] / shape_viejo[0]

0.9488

### Predicción

In [73]:
ids = df_predic["CUSTOMER_ID"]
df_predic.drop(columns=ls_drop, inplace=True)

In [74]:
pred1 = pipe_ss.predict(df_predic)

In [75]:
pred2 = pipe_h.predict(df_predic)

In [76]:
df_resultado_cv = pd.DataFrame({
    "CARDHOLDER_ID": ids,
    "y_hat": pred1
})

In [77]:
df_resultado_hp = pd.DataFrame({
    "CARDHOLDER_ID": ids,
    "y_hat": pred2
})

In [83]:
df_resultado_cv.to_csv("Respuestas/MoctezumaRamirezDiegoRafael_PAYAMT1.csv", index=False)

In [79]:
df_resultado_cv

,CARDHOLDER_ID,y_hat
0,17391,10675.144113
1,14756,314.440523
3,20929,2354.823286
4,6533,4198.643692
5,12435,-1666.820410
...,...,...
1870,12040,-14.401009
1871,3318,13094.275315
1872,2124,5648.501733
1873,13129,4541.550502


In [84]:
df_resultado_hp.to_csv("Respuestas/MoctezumaRamirezDiegoRafael_PAYAMT1.csv", index=False)

In [81]:
df_resultado_hp

,CARDHOLDER_ID,y_hat
0,17391,10602.573988
1,14756,177.920479
3,20929,2272.632804
4,6533,4246.878321
5,12435,-1465.188536
...,...,...
1870,12040,1.633494
1871,3318,11228.418534
1872,2124,4965.548069
1873,13129,4346.016276
